# 02_horvath_python

This notebook implements the **Horvath DNA methylation age (DNAmAge) model**
in Python.

Objectives:
- load the processed CpG × Sample β-value matrix
- load Horvath CpG coefficients extracted from the R implementation
- compute DNAmAge using the original Horvath transformation
- export predicted DNAmAge values for downstream comparison

This notebook focuses on **explicit model implementation**, not evaluation.

## 1. Setup

Load required libraries and define project paths.

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("..").resolve()
PROC_DIR = ROOT / "data" / "processed"

## 2. Load β-value matrix

Load the CpG × Sample DNA methylation β-value matrix generated in
`01_preprocess_GSE40279.ipynb`.

In [3]:
beta = pd.read_csv(
    PROC_DIR / "GSE40279_beta_for_R.csv",
    index_col=0
)

beta.shape

(473034, 656)

## 3. Load Horvath CpG coefficients

Load the CpG coefficients and intercept extracted at runtime from
the R implementation (`wateRmelon::agep`).

This ensures full consistency with the original Horvath model.

In [4]:
coef_df = pd.read_csv(
    PROC_DIR / "horvath_coefficients_from_runtime.csv",
    index_col=0
)

coef_df.head(), coef_df.shape

(                 Coef
 CpG                  
 (Intercept)  0.695507
 cg00075967   0.129337
 cg00374717   0.005018
 cg00864867   1.599764
 cg00945507   0.056852,
 (354, 1))

## 4. Align CpGs between β matrix and coefficient table

Restrict the β-value matrix to CpGs used by the Horvath model
and ensure consistent ordering.

In [6]:
# Separate intercept and CpG coefficients
intercept = coef_df.loc["(Intercept)", "Coef"]
coef_cpg = coef_df.drop("(Intercept)")

# Align CpGs
common_cpgs = beta.index.intersection(coef_cpg.index)

beta_sub = beta.loc[common_cpgs]
coef_sub = coef_cpg.loc[common_cpgs]

beta_sub.shape, coef_sub.shape

((353, 656), (353, 1))

## 5. Linear prediction (Horvath model)

The Horvath model first computes a **linear predictor**:

z = intercept + Σ (β_i × w_i)

where:
- β_i are CpG methylation values
- w_i are CpG-specific coefficients

In [8]:
# Linear predictor (z)
z = intercept + beta_sub.T.values @ coef_sub["Coef"].values

z = pd.Series(z, index=beta.columns, name="z")
z.head()

GSM989827    1.531483
GSM989828    2.640296
GSM989829    2.028709
GSM989830    1.630937
GSM989831    2.045106
Name: z, dtype: float64

## 6. Inverse transformation to DNAmAge

The Horvath model does **not** predict age directly.

Instead, the linear predictor `z` is converted to DNAmAge (in years)
using a piecewise inverse transformation defined in the original paper.

In [9]:
ADULT_AGE = 20.0

def horvath_invF(z):
    """
    Inverse transformation from linear predictor to DNAmAge
    as defined in Horvath (2013).
    """
    z = np.asarray(z, dtype=float)
    age = np.empty_like(z)

    m = z < 0
    age[m]  = (ADULT_AGE + 1.0) * np.exp(z[m] + np.log(ADULT_AGE + 1.0)) - 1.0
    age[~m] = (ADULT_AGE + 1.0) * z[~m] + ADULT_AGE
    return age

In [10]:
dnam_py = pd.Series(
    horvath_invF(z.values),
    index=z.index,
    name="DNAmAge_python"
)

dnam_py.head()

GSM989827    52.161142
GSM989828    75.446216
GSM989829    62.602883
GSM989830    54.249671
GSM989831    62.947226
Name: DNAmAge_python, dtype: float64

## 7. Export predicted DNAmAge

Save Python-based DNAmAge predictions for downstream comparison
with the R implementation.

In [11]:
out_path = PROC_DIR / "GSE40279_DNAmAge_python.csv"
dnam_py.to_csv(out_path)

out_path

PosixPath('/Volumes/Extreme_Pro/mac/Coding/m_clock/methylation-clock-replication/data/processed/GSE40279_DNAmAge_python.csv')

## Notes

This notebook provides a **transparent Python implementation** of the
Horvath DNAmAge model, including the original inverse age transformation.

Validation and comparison with the R implementation
(`wateRmelon::agep`) are performed in:

- `03_compare_with_agep.ipynb`